## Setup

This notebook needs two packages that Colab does not preinstall
(`pytransform3d` and `cvxopt`), and it reads the calibration images and the
rectangle images from this repository. The cell below handles both when it
detects Colab, and does nothing when you run the notebook locally.

In [ ]:
# ---------------------------------------------------------------------------
# Colab setup. Installs the two packages Colab does not ship, and clones this
# repository so that the calibration images and the rectangle images are
# available. Running locally, this cell does nothing but report the directory.
# ---------------------------------------------------------------------------
import os
import sys

REPO_URL  = "https://github.com/eraldoribeiro/camera-pose-from-planes-solution.git"
REPO_NAME = "camera-pose-from-planes-solution"

if "google.colab" in sys.modules:
    print("Running in Google Colab")

    # Not part of the standard Colab image:
    !pip install -q pytransform3d cvxopt

    if not os.path.exists(REPO_NAME):
        !git clone -q {REPO_URL}

    # Work from inside the repo, so the relative paths further down resolve.
    if os.path.basename(os.getcwd()) != REPO_NAME:
        os.chdir(REPO_NAME)

print("working directory :", os.getcwd())
n_cal = len([f for f in os.listdir("calibration") if f.endswith(".jpg")]) \
        if os.path.isdir("calibration") else 0
print("calibration images:", n_cal)
print("rectangle images  :", len([f for f in os.listdir(".")
                                  if f.startswith("dory_rectangle")]))

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection, Line3DCollection
from cvxopt import matrix, printing
from scipy.spatial.transform import Rotation as R
from mpl_toolkits.axes_grid1 import ImageGrid
import pytransform3d.camera as pc
import pytransform3d.transformations as pt
import fnmatch
import os

In [ ]:
#%matplotlib qt
%matplotlib inline

In [ ]:
import os
filename = "plot_image_grid.py"
url = "https://raw.githubusercontent.com/facebookresearch/pytorch3d/main/docs/tutorials/utils/plot_image_grid.py"
if not os.path.exists(filename):
    !wget {url}

# Camera calibration

Use OpenCV to calibrate the camera so that we can undistort images.

https://docs.opencv.org/4.x/dc/dbb/tutorial_py_calibration.html


#### Utility function (display images in a grid)

The following function displays a set of images as a M x N rectangular grid.

In [ ]:
def display_images_in_grid(images, nrows, ncols, sx, sy):
    # Display all images in a nrows x ncols tight grid

    fig = plt.figure(figsize=(sx, sy))
    grid = ImageGrid(fig, 111,  # similar to subplot(111)
                     nrows_ncols=(nrows, ncols),  # creates 2x2 grid of axes
                     axes_pad=0.1  # pad between axes in inch.
                     )

    for ax, im in zip(grid, images):
        # Iterating over the grid returns the Axes.
        ax.imshow(im)
        ax.set_axis_off()

    plt.show()
    
    return

#### Function to estimate the calibration matrix
The following function calculates the calibration given a set of images of a checkerboard pattern seen from different view points.


In [ ]:
def obtain_camera_matrix_distcoefs():

    # Criteria for termination of the iterative process of corner refinement
    terminate_criteria = (cv.TERM_CRITERIA_EPS +
                          cv.TERM_CRITERIA_MAX_ITER, 30, 0.001)

    # Number of corners to look for in the chessboard
    horizontal_corners = 9
    vertical_corners = 6

    # Create an array of object points. These are the 3D coordinates of the corners in the real world
    object_points = np.zeros(
        (horizontal_corners * vertical_corners, 3), np.float32)
    object_points[:, :2] = np.mgrid[0:horizontal_corners,
                                    0:vertical_corners].T.reshape(-1, 2)
    
    # Count the number of calibration images to be used
    num_images = len(fnmatch.filter(os.listdir("calibration/"), '*.jpg'))
    print('File Count:', num_images)    

    # Load the image, scale it, and make it grayscale
    cal_imgs = []
    gray_imgs = []
    for i in range(0, num_images):
        image = cv.imread("calibration/image_{}.jpg".format(i))
        scaled = cv.resize(image, (0, 0), fx=0.2, fy=0.2)
        color_fixed = cv.cvtColor(scaled, cv.COLOR_BGR2RGB)
        gray = cv.cvtColor(scaled, cv.COLOR_BGR2GRAY)
        cal_imgs.append(color_fixed)
        gray_imgs.append(gray)

    # Find the corners in the chessboards
    ret = []
    corners = []
    for i in range(0, num_images):
        ret_temp, corners_temp = cv.findChessboardCorners(
            gray_imgs[i], (horizontal_corners, vertical_corners), None)

        if ret_temp != True:
            raise Exception("Chessboard not found in image {}".format(i))

        ret.append(ret_temp)
        corners.append(corners_temp)

    ###------------------------------------------------------------------###
    #  Refine corners and display results                                  #
    ###------------------------------------------------------------------###        
        
    # Refine the corners
    corners_refined = []
    for i in range(0, num_images):
        corners_refined_temp = cv.cornerSubPix(
            gray_imgs[i], corners[i], (11, 11), (-1, -1), terminate_criteria)
        corners_refined.append(corners_refined_temp)
   
    # Calibration images showing refined corners 
    refined_corner_images = [cv.drawChessboardCorners(cal_imgs[i], 
                                                      (horizontal_corners, vertical_corners), 
                                                      corners_refined[i], ret[i])
                      for i in range(0,num_images)]     
    
    ###------------------------------------------------------------------###
    #  Calibrate camera                                                    #
    ###------------------------------------------------------------------###        
    
    # Calculate camera calibration parameters
    reprojection_error, camera_matrix, distortion_coefficients, _, _ = cv.calibrateCamera(
        [object_points, object_points, object_points], corners_refined, gray.shape[::-1], None, None)

    # Refine the camera matrix
    alpha = 0
    refined_camera_matrix, roi = cv.getOptimalNewCameraMatrix(
        camera_matrix, distortion_coefficients, gray.shape[::-1], alpha, gray.shape[::-1])


    # Undistort the images
    undistorted_images = []
    for i in range(0, num_images):
        undistorted_image = cv.undistort(cal_imgs[i], 
                                         camera_matrix, 
                                         distortion_coefficients, None, refined_camera_matrix)
        undistorted_images.append(undistorted_image)
        
    ###------------------------------------------------------------------###
    #  Calculate and show undistorted images                                        #
    ###------------------------------------------------------------------###        
        
    # Crop the images
    undistorted_images_cropped = []
    for i in range(0, num_images):
        x, y, w, h = roi
        undistorted_image_cropped = undistorted_images[i][y:y+h, x:x+w]
        undistorted_images_cropped.append(undistorted_image_cropped)

    # Display images as a grid
    display_images_in_grid(undistorted_images_cropped, 1, num_images, 20, 20)    
   
    
    return camera_matrix, refined_camera_matrix, distortion_coefficients

In [ ]:
import numpy as np
import cv2 as cv
import glob
 
# termination criteria
criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 30, 0.001)
 
# prepare object points, like (0,0,0), (1,0,0), (2,0,0) ....,(6,5,0)
objp = np.zeros((6*9,3), np.float32)
objp[:,:2] = np.mgrid[0:9,0:6].T.reshape(-1,2)
 
# Arrays to store object points and image points from all the images.
objpoints = [] # 3d point in real world space
imgpoints = [] # 2d points in image plane.
 
images = glob.glob('calibration/*.jpg')
 
rgb_images = [] 
for fname in images:
    img = cv.imread(fname)
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
 
    # Find the chess board corners
    ret, corners = cv.findChessboardCorners(gray, (9,6), None)
 
    # If found, add object points, image points (after refining them)
    if ret == True:
        objpoints.append(objp)
 
        corners2 = cv.cornerSubPix(gray,corners, (11,11), (-1,-1), criteria)
        imgpoints.append(corners2)
 
        # Draw and display the corners
        cv.drawChessboardCorners(img, (9,6), corners2, ret)

        # store rgb image
        color_fixed = cv.cvtColor(img, cv.COLOR_BGR2RGB)
        rgb_images.append(color_fixed)


        # # display image using plt
        # plt.imshow(img)
        # plt.show()

In [ ]:
from plot_image_grid import image_grid

image_grid(rgb_images, rows=4, cols=5, rgb=True, show_axes=False)

In [ ]:
# Calibrate camera 
ret, mtx, dist, rvecs, tvecs = cv.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)

In [ ]:
# Estimate the camera matrix and lens distortion
# camera_matrix, refined_camera_matrix, distortion_coefficients = obtain_camera_matrix_distcoefs()

# Matrix of intrinsic parameters
print('---------------------------------------------------')
print('Matrix of intrinsic parameters')
print('---------------------------------------------------')
print('Lambda = ')
printing.options['dformat'] = '%.2f'
printing.options['width'] = -1
print(matrix(mtx))

### Feature detection and matching

Here, we will use a set of four feature pairs corresponding to the corner of rectangles on the images. 

In [ ]:
# Make a list of input images of the rectangles. 
raw_images = [cv.imread('dory_rectangle{}.jpg'.format(i)) for i in range(0, 4)]
# Convert all images in the list from BGR to RGB to show correct colors. 
raw_images = [cv.cvtColor(img, cv.COLOR_BGR2RGB) for img in raw_images]

########################################################################
# In this example, the aw images are already corrected 
# for lens distortion. When using images from your own camera, you 
# might need to correct for lens distortion. 
# To do that, uncomment the following steps and adapt the names of variables. 
########################################################################

# Undistort the images using the refined camera matrix (removes lens distortion)
#undistorted_images = [cv.undistort(img, camera_matrix, distortion_coefficients, None, refined_camera_matrix) for img in scaled_images]

# Display images as a grid
display_images_in_grid(raw_images, 2, 2, 15, 15)

# Estimate homography transformation

Estimate the matrix $\Phi$ of the planar projective transformation.


### Input data: reference points and image (feature) points
The input data for the homography calculation are a set of pairs of coordinates. Each pair is formed by a set of landmarks on the reference pattern and a set of correspoding feature points on the image of the reference pattern. 

In [ ]:
ref_points = [[[  0,   0]],
              [[  0, 511]],
              [[511, 511]],
              [[511,   0]]]

img_points =[
        [ # image 0
        [[305, 206]],
        [[298, 374]],
        [[471, 382]],       
        [[476, 207]]
        ], 
        [ # image 1
        [[363, 320]],
        [[302, 446]],
        [[458, 493]],
        [[504, 349]]
        ],
        [ # image 2
        [[534, 308]],
        [[431, 195]],
        [[292, 275]],
        [[394, 411]]
        ],
        [ # image 3
        [[291, 434]],
        [[500, 369]],
        [[423, 216]],
        [[243, 246]]
        ]
    ]

## Show the labeled corners on the images

In [ ]:
labelled_points = raw_images

for i in range(0,4):
    for j in range(0,4):
        # Draw circles at the image features
        labelled_points[i] = cv.circle(labelled_points[i],
                           (np.int32(img_points[i][j][0][0]),np.int32(img_points[i][j][0][1])), 
                           8, (255,0,0), -1)
        
        # Draw text labels at the image features 
        labelled_points[i] = cv.putText(labelled_points[i],'{}'.format(j), 
                          (np.int32(img_points[i][j][0][0]+15),np.int32(img_points[i][j][0][1]+10)), 
                          cv.FONT_HERSHEY_SIMPLEX, 1, (0,0,0), 2)

display_images_in_grid(labelled_points, 2, 2, 15, 15)

In [ ]:
# Find the homography matrix for each image
#
# I'm using the built-in function instead of computing it myself. The built-in function
# filters out the outliers.
homography_matrices = []
matches_mask = []

printing.options['dformat'] = '%.3f'
printing.options['width'] = -1

for pts, idx in zip(img_points, range(0,4)):
    h, mask = cv.findHomography(np.array(ref_points), np.array(pts), cv.RANSAC, 5.0)
    
    print(f"H({idx}) = ")
    print(matrix(h))
    homography_matrices.append(h)
    matches_mask.append(mask.ravel().tolist())

# Extract extrinsic parameters

Factorize the homography matrix to extract the camera rotation and translation.

## The equations

For a **planar** target we place the world frame on the plane, so every object
point has $w=0$. The pinhole model then collapses from a $3\times4$ projection to
a $3\times3$ homography (see the previous notes):

$$
\begin{align}
\lambda\,\tilde{\bf x}
   \;=\; \Lambda\begin{bmatrix}\Omega & \boldsymbol{\tau}\end{bmatrix}\tilde{\bf w}
   \;\;\xrightarrow{\;w=0\;}\;\;
\lambda\,\tilde{\bf x}
   \;=\; \underbrace{\Lambda\begin{bmatrix}{\bf r}_1 & {\bf r}_2 & \boldsymbol{\tau}\end{bmatrix}}_{\Phi}
   \begin{bmatrix}u\\ v\\ 1\end{bmatrix},
\tag{1}
\end{align}
$$

where ${\bf r}_1,{\bf r}_2$ are the first two columns of the rotation matrix
$\Omega$. So the homography we estimated is, **up to an unknown scale**,

$$
\begin{align}
\Phi \;\simeq\; \Lambda\begin{bmatrix}{\bf r}_1 & {\bf r}_2 & \boldsymbol{\tau}\end{bmatrix}.
\tag{2}
\end{align}
$$

### Step 1 --- remove the intrinsics

Multiplying by $\Lambda^{-1}$ strips the camera's internal parameters and leaves
the pose, still scaled by the unknown $\lambda$:

$$
\begin{align}
\tilde{\Phi} \;=\; \Lambda^{-1}\Phi
   \;=\; \lambda\begin{bmatrix}{\bf r}_1 & {\bf r}_2 & \boldsymbol{\tau}\end{bmatrix}
   \;=\; \begin{bmatrix}\lambda{\bf r}_1 & \lambda{\bf r}_2 & \lambda\boldsymbol{\tau}\end{bmatrix}.
\tag{3}
\end{align}
$$

Write its columns as $\tilde{\Phi} = [\,\tilde{\bf h}_1\ \tilde{\bf h}_2\ \tilde{\bf h}_3\,]$.

### Step 2 --- recover the scale $\lambda$

The columns of a rotation matrix are **unit vectors**, so
$\|\lambda{\bf r}_1\| = |\lambda|$. That gives us the scale for free:

$$
\begin{align}
\lambda \;=\; \frac{1}{\|\tilde{\bf h}_1\|} \;=\; \frac{1}{\|\tilde{\bf h}_2\|}
\qquad\text{in practice}\qquad
\lambda \;=\; \frac{2}{\|\tilde{\bf h}_1\| + \|\tilde{\bf h}_2\|},
\tag{4}
\end{align}
$$

averaging the two estimates because noise makes them differ slightly.

### Step 3 --- fix the sign

$\Phi$ and $-\Phi$ describe the *same* homography, so Equation 4 determines
$\lambda$ only up to sign. We resolve it with physics: the target must be **in
front of** the camera, i.e. it must have positive depth,

$$
\begin{align}
\tau_z \;=\; \big(\lambda\tilde{\bf h}_3\big)_z \;>\; 0 .
\tag{5}
\end{align}
$$

If it comes out negative, flip the sign of $\lambda$.

### Step 4 --- build the rotation

$$
\begin{align}
{\bf r}_1 = \lambda\tilde{\bf h}_1, \qquad
{\bf r}_2 = \lambda\tilde{\bf h}_2, \qquad
{\bf r}_3 = {\bf r}_1 \times {\bf r}_2, \qquad
\boldsymbol{\tau} = \lambda\tilde{\bf h}_3 .
\tag{6}
\end{align}
$$

The third column is *computed*, not measured: a planar target carries no
information about the plane normal, so ${\bf r}_3$ has to come from the
right-handedness of the frame.

### Step 5 --- project onto a valid rotation

With real, noisy data the matrix $Q = [\,{\bf r}_1\ {\bf r}_2\ {\bf r}_3\,]$ is
close to a rotation but not exactly one. The **nearest** rotation in the
Frobenius sense is obtained from the SVD $Q = ULV^\mathsf{T}$:

$$
\begin{align}
\Omega \;=\; U
\begin{bmatrix} 1 & 0 & 0\\ 0 & 1 & 0\\ 0 & 0 & \det(UV^\mathsf{T})\end{bmatrix}
V^\mathsf{T},
\tag{7}
\end{align}
$$

where the $\det(UV^\mathsf{T})$ term guarantees $\det\Omega = +1$, so that
$\Omega \in \mathrm{SO}(3)$ and we get a rotation rather than a reflection.


In [ ]:
def extractExtrinsicParametersFromHomography(Phi, Lambda_inverse):
    """
    Factorize a planar homography to recover the camera pose.

    Implements Equations (3)-(7) of the notes above.

    Input arguments
    ---------------
        Phi:            3x3 homography matrix, mapping the plane to the image
        Lambda_inverse: 3x3 inverse of the intrinsic matrix

    Returns
    -------
        Omega: 3x3 rotation matrix, in SO(3)
        tau:   3x1 translation vector (world origin in camera coordinates)
    """

    # --- Step 1: remove the intrinsics.  Phi_tilde = [lambda*r1, lambda*r2, lambda*tau]
    Phi_tilde = Lambda_inverse @ Phi
    h1, h2, h3 = Phi_tilde[:, 0], Phi_tilde[:, 1], Phi_tilde[:, 2]

    # --- Step 2: the columns of a rotation are unit vectors, so ||lambda*r_i|| = |lambda|.
    #     Average the two estimates, which noise makes slightly different.
    scaling_factor = 2.0 / (np.linalg.norm(h1) + np.linalg.norm(h2))

    # --- Step 3: Phi and -Phi are the same homography, so the sign of lambda is
    #     ambiguous. Resolve it physically: the target must be IN FRONT of the
    #     camera, so its depth must be positive.
    if (scaling_factor * h3)[2] < 0:
        scaling_factor = -scaling_factor

    # --- Step 4: rotation columns and translation.
    r1 = scaling_factor * h1
    r2 = scaling_factor * h2
    r3 = np.cross(r1, r2)          # a planar target says nothing about the normal
    tau = scaling_factor * h3

    # --- Step 5: project onto the nearest true rotation (Frobenius sense).
    #     det(U @ Vt) keeps the result a rotation instead of a reflection.
    Q = np.column_stack([r1, r2, r3])
    U, L, Vt = np.linalg.svd(Q)
    Omega = U @ np.diag([1.0, 1.0, np.linalg.det(U @ Vt)]) @ Vt

    return Omega, tau

### A note on image resolution

The intrinsic matrix is only valid at the resolution it was **calibrated** at.
The calibration images here are $640\times480$, but the test images of the
rectangle are $800\times600$, so $\Lambda$ has to be rescaled before it can be
used with them. Scaling the image by $s_x$ horizontally and $s_y$ vertically
scales the corresponding rows of $\Lambda$:

$$
\begin{align}
\Lambda' \;=\;
\begin{bmatrix} s_x & 0 & 0\\ 0 & s_y & 0\\ 0 & 0 & 1\end{bmatrix}
\Lambda,
\qquad
s_x = \frac{800}{640}, \quad s_y = \frac{600}{480}.
\tag{8}
\end{align}
$$

Both the focal lengths and the principal point move; the skew and the bottom row
do not. Skipping this step leaves the focal length about 20\% too short, which
tilts every recovered pose.


In [ ]:
# --- The intrinsics were calibrated at the resolution of the CALIBRATION
#     images, which is not the resolution of the rectangle images. Rescale
#     Lambda before using it (Equation 8).
calib_height, calib_width = gray.shape[:2]                 # calibration images
image_height, image_width = raw_images[0].shape[:2]        # rectangle images

scale = np.diag([image_width / calib_width,
                 image_height / calib_height,
                 1.0])
Lambda_test = scale @ mtx

print(f"calibration images : {calib_width} x {calib_height}")
print(f"rectangle images   : {image_width} x {image_height}")
print("\nLambda (as calibrated) =\n", np.round(mtx, 2))
print("\nLambda rescaled to the rectangle images =\n", np.round(Lambda_test, 2))

# Calculate the inverse of the intrinsic camera matrix
inverse_intrinsic_camera_matrix = np.linalg.inv(Lambda_test)

camera_rotations = []
camera_translations = []
for homography in homography_matrices:
    # Extract extrinsic parameters from homography
    rotation_matrix, translation = extractExtrinsicParametersFromHomography(
        homography, inverse_intrinsic_camera_matrix)

    camera_rotations.append(rotation_matrix)
    camera_translations.append(translation)

# Reference frame
camera_rotations.append(np.eye(3))
camera_translations.append(np.array([0, 0, 0]))

# Print the camera poses, and check each one is a valid rotation
for i, (rotation, translation) in enumerate(zip(camera_rotations, camera_translations)):
    is_reference = np.allclose(rotation, np.eye(3)) and np.allclose(translation, 0)
    print("--------------------------------------")
    print("Camera pose {}{}:".format(i, "  (world reference frame)" if is_reference else ""))
    print("--------------------------------------\n")
    print("Rotation:")
    print(matrix(rotation))
    print("Translation:")
    print(matrix(translation))
    if is_reference:
        print()
        continue
    print(f"\n  det(Omega) = {np.linalg.det(rotation):+.6f}   "
          f"(must be +1)")
    print(f"  orthonormal: {np.allclose(rotation.T @ rotation, np.eye(3), atol=1e-9)}")
    print(f"  depth tau_z = {np.asarray(translation).ravel()[2]:+.1f}   "
          f"(must be > 0: the target is in front of the camera)\n")

### Draw the cameras (change coordinate frames)
To draw the cameras, we need to keep in mind that the camera-pose matrices that have been estimated from the homography transformations are written in terms of camera coordinate frames. As a result, we cannot use those matrices directly to plot the cameras. If we do, the cameras will not display properly. Instead, we need to calculate the inverse of the camera-pose matrices to obtain the camera pose written with respect to global coordinates. 

The camera pose is represented by a rigid-body transformation, i.e.: 

$$
\begin{align}
    T = 
    \begin{bmatrix}
        R & t \\
        {\bf 0} & 1
    \end{bmatrix},
\end{align}
$$

which has the following inverse: 

$$
\begin{align}
    T^{-1} = 
    \begin{bmatrix}
        R^\mathsf{T} & -R^\mathsf{T}t \\
        {\bf 0} & 1
    \end{bmatrix}.
\end{align}
$$

In [ ]:
import pytransform3d.camera as pc
import pytransform3d.transformations as pt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# ---------------------------------------------------------------------------
# Everything the plot needs is defined here.
# ---------------------------------------------------------------------------
intrinsic_matrix      = Lambda_test                    # K for the 800x600 images
sensor_size           = np.array([image_width, image_height])
virtual_image_distance = 300                           # in world units (pattern = 511)

fig = plt.figure(figsize=(9, 7))
ax  = fig.add_subplot(111, projection="3d")

# --- the calibration plane itself, so we can see what the cameras look at ---
square = np.array([[0, 0, 0], [511, 0, 0], [511, 511, 0], [0, 511, 0]], dtype=float)
ax.add_collection3d(Poly3DCollection([square], facecolor="#c9d8ec",
                                     edgecolor="#1f4e9c", alpha=0.75))

# --- the world frame, which sits on the plane -------------------------------
pt.plot_transform(ax, A2B=np.eye(4), s=250)

# --- one frustum per camera -------------------------------------------------
for cam_idx, (Omega, tau) in enumerate(zip(camera_rotations, camera_translations)):

    Omega = np.asarray(Omega).reshape(3, 3)
    tau   = np.asarray(tau).reshape(3, 1)

    # Skip the identity "reference frame" entry appended to the lists earlier;
    # the world frame is already drawn above.
    if np.allclose(Omega, np.eye(3)) and np.allclose(tau, 0):
        continue

    # The estimated pose maps WORLD -> CAMERA. To draw the camera we need the
    # inverse, i.e. the camera expressed in world coordinates:
    #     T_cw = [ Omega^T   -Omega^T tau ]
    #            [   0             1      ]
    cam2world = np.vstack([np.block([Omega.T, -Omega.T @ tau]),
                           np.array([0, 0, 0, 1])])

    camera_centre = (-Omega.T @ tau).ravel()
    optical_axis  = Omega.T @ np.array([0.0, 0.0, 1.0])   # camera +z, in world

    print(f"camera {cam_idx}:  centre = {np.round(camera_centre, 1)}"
          f"   optical axis = {np.round(optical_axis, 3)}")

    pt.plot_transform(ax, A2B=cam2world, s=150)
    pc.plot_camera(ax, cam2world=cam2world, M=intrinsic_matrix,
                   sensor_size=sensor_size,
                   virtual_image_distance=virtual_image_distance)
    ax.text(*camera_centre, f"  cam{cam_idx}", fontsize=9)

ax.set_xlim(-800, 1400)
ax.set_ylim(-800, 1400)
ax.set_zlim(-2200, 400)
ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
ax.view_init(22, -70)
ax.set_title("Recovered camera poses")
plt.show()